# Run PureDefense Class

In [3]:
from PureDefense import PureDefense

import os
import argparse

from utils.utils import *

from utils.utils_purify import get_poisons, ImageListDataset, save_poisons

## Args

In [5]:
os.environ['XRT_TPU_CONFIG'] = 'localservice;0;localhost:51011'

# ======== arg parser =================================================
parser = argparse.ArgumentParser(description='PyTorch Poison Attack')

### Setup Arguments ###
parser.add_argument('--remote_user', type=str, help='username for the remote server (TPU only, else pass in full directory args below)')
# parser.add_argument('--num_proc', type=int, default=8, help='number of processes for TPU')
parser.add_argument('--device_type', default='xla', type=str, choices=['xla','cuda','cpu','mps'],help='device type to use')
parser.add_argument('--seed', default=11, type=int,help='seed for reproducibility')
parser.add_argument('--verbose','--v', default=False, action='store_true',help='print out additional information when running')
parser.add_argument('--data_dir', default='/home/data/', type=str, help='path to the data directory')
parser.add_argument('--num_proc', type=int, default=8, help='number of processes for TPU')

### Experiment Arguments ###
parser.add_argument('--dataset', default='cifar10', type=str, choices=['cifar10','cinic10','stl10','tinyimagenet'],help='dataset to use')

### Purification Arguments ###

parser.add_argument('--purify_reps', default=1, type=int, help='number of purification repetitions (when using both EBM and Diffusion)')

# EBM Arguments 
args_ebm = parser.add_argument_group('EBM')
args_ebm.add_argument('--ebm_model', default='EBMSNGAN32', type=str, choices=['EBM_Small','EBMSNGAN32','EBMSNGAN128','EBMSNGAN256'],help='type of EBM model to use')
args_ebm.add_argument('--ebm_name', default='ebm_cifar10_45k', type=str, help='path to the EBM model including train dataset')
args_ebm.add_argument('--ebm_nf', default=128, type=int,  help='number of filters for the ebm model')
args_ebm.add_argument('--ebm_lang_steps', default=150, type=int, help='number of langevin steps')
args_ebm.add_argument('--ebm_lang_temp', default=1e-4, type=float, help='langevin temperature')

# Diffusion Arguments
args_diff = parser.add_argument_group('Diffusion')
args_diff.add_argument('--diff_model', default='DDPM_UNET_EBM', type=str, choices=['DDPM_UNET','DDPM_UNET_EBM'],help='type of diffusion model to use')
args_diff.add_argument('--diff_name', default='mcmc_steps1600_bank_fixer_cosine', type=str, help='path to the diffusion model')
args_diff.add_argument('--diff_nf', default=128, type=int,  help='number of filters for the unet model')
args_diff.add_argument('--diff_train_steps', default=1000, type=int, help='training t-steps for diffuion model')
args_diff.add_argument('--diff_output', default='epsilon', type=str, choices=['epsilon','start_x'],  help='diffusion model output')
args_diff.add_argument('--diff_schedule', default='cosine', type=str, choices=['linear','cosine'], help='t schedule')
args_diff.add_argument('--diff_purify_steps', default=10, type=int,  help='number of purify t-steps for the unconditional diffuion model')
args_diff.add_argument('--diff_eta', default=0, type=int,  help='ddpm 1 or ddim 0 for the sampling of the 1000 tstep fixer')
    

### Poison Arguments ###
parser.add_argument('--poison_type', default=None, type=str, choices=['Narcissus', 'Gradient_Matching','BullseyePolytope','BullseyePolytope_Bench'],help='type of poison to generate')
parser.add_argument('--poison_mode', default='from_scratch', type=str, choices=['from_scratch','transfer'],help='mode of attack')
parser.add_argument('--noise_sz_narcissus', default=32, type=int, help='size of the noise trigger for Narcissus')
parser.add_argument('--noise_eps_narcissus', default=8, type=int, help='epsilon for the noise trigger for Narcissus')
parser.add_argument('--num_images_narcissus', default=500, type=int, help='number of poisoned images generated')
parser.add_argument('--random_imgs_narcissus', default=False, action='store_true', help='use random images for narcissus')
parser.add_argument('--iters_bp', default=800, type=int,help='iterations for making poison')
parser.add_argument('--num_images_bp', default=50, type=int,help='number of poisoned images generated')
parser.add_argument('--net_repeat_bp', default=1, type=int, help='number of times to repeat the network for methods BP-1, BP-3, BP-5')
parser.add_argument('--num_per_class_bp', default=50, type=int, help='num of samples per class for re-training, or the poison dataset')

# Parse the arguments
args = parser.parse_args(['--remote_user','sunaybhat','--v'])

device = get_device(args.device_type)
set_seed(args.seed, device, args.device_type)

# Setup directories for remote server
if args.remote_user is not None:
    args.data_dir = args.data_dir.replace('/home',f'/home/{args.remote_user}')

## Get Data to Purify

In [6]:
# Get the data loader and number of target indices
if args.poison_type is None:
    train_data = torchvision.datasets.CIFAR10(root=args.data_dir, train=True, download=True, transform=torchvision.transforms.ToTensor())
    train_loader = torch.utils.data.DataLoader(train_data, batch_size=128, shuffle=False, num_workers=4)
    target_indices = 1
    purify_pbar = True
else:
    poison_tuple_list, poison_indices, target_mask_label = get_poisons(args,0)
    train_loader = torch.utils.data.DataLoader(ImageListDataset(poison_tuple_list), batch_size=128, shuffle=False, num_workers=4)

    if args.poison_type == 'Narcissus':
        target_indices = 10
    elif args.poison_type == 'Gradient_Matching':
        target_indices = 100
    elif args.poison_type == 'BullseyePolytope':
        if args.num_images_bp == 50:
            target_indices = 48 # 48 images in the dataset from original paper
        else:
            target_indices = 50
    elif args.poison_type == 'BullseyePolytope_Bench':
        target_indices = 100

    purify_pbar = False

Files already downloaded and verified


In [9]:
# Get diff and ebm model paths
ebm_path = os.path.join(args.data_dir,'models',args.ebm_model,args.dataset,args.ebm_name+'.pt')
diff_path = os.path.join(args.data_dir,'models',args.diff_model,args.dataset,args.diff_name+'.pt')
                          
PurifyClass = PureDefense(device,args.device_type,
                        ebm_type=args.ebm_model,ebm_path=ebm_path,ebm_nf=args.ebm_nf,
                        diff_type=args.diff_model,diff_path=diff_path,diff_nf=args.diff_nf,
                        diff_schedule=args.diff_schedule,
                        diff_train_steps=args.diff_train_steps,diff_output=args.diff_output,
                        img_sz=32,verbose=args.verbose)

Loaded EBMSNGAN32 from /home/sunaybhat/data/models/EBMSNGAN32/cifar10/ebm_cifar10_45k.pt
1000
Loaded DDPM_UNET_EBM from /home/sunaybhat/data/models/DDPM_UNET_EBM/cifar10/mcmc_steps1600_bank_fixer_cosine.pt


## Purify (Baseline Data or all Poisons of a type) and Save

In [11]:
if purify_pbar is False:
    pbar = tqdm(total=target_indices, desc='Purifying Poisoned Data')

for i,args.target_index in enumerate(range(target_indices)):

    ### Get Data to Purify ###
    if args.poison_type is None:
        train_data = torchvision.datasets.CIFAR10(root=args.data_dir, train=True, download=True, transform=torchvision.transforms.ToTensor())
        train_loader = torch.utils.data.DataLoader(train_data, batch_size=128, shuffle=False, num_workers=4)
    else:
        poison_tuple_list, poison_indices, target_mask_label = get_poisons(args,0)
        train_loader = torch.utils.data.DataLoader(ImageListDataset(poison_tuple_list), batch_size=128, shuffle=False, num_workers=4)

    ### Purify the dataset ###
    purified_data = PurifyClass.purify(train_loader,ebm_lang_steps=args.ebm_lang_steps,ebm_lang_temp=args.ebm_lang_temp,
                    diff_steps=args.diff_purify_steps, diff_eta=args.diff_eta,
                    purify_reps=1,pbar=purify_pbar)

    ### Save the purified data ###
    data_key = ''
    if args.ebm_lang_steps > 0:
        data_key += f'{args.ebm_model}_{args.ebm_name}_nf{args.ebm_nf}_{args.ebm_lang_steps}Steps_T{args.ebm_lang_temp}'
    if args.diff_purify_steps > 0:
        data_key += f'_{args.diff_model}_{args.diff_name}_nf{args.diff_nf}_beta[{args.diff_train_steps}_{args.diff_schedule}]_{args.diff_purify_steps}Steps_{args.diff_eta}eta'
    if args.ebm_lang_steps > 0 and args.diff_purify_steps > 0:
        data_key += f'_reps{args.purify_reps}'
    
    if data_key == '':
        data_key = 'baseline'

    if args.poison_type is None:
        if not os.path.exists(os.path.join(args.data_dir,'PureDefense',args.dataset)):
            os.makedirs(os.path.join(args.data_dir,'PureDefense',args.dataset))
        torch.save(train_data,os.path.join(args.data_dir,'PureDefense',args.dataset,f'{data_key}.pt'))
    else:
        save_dir = save_poisons(args,train_data, poison_indices, target_mask_label, data_key)

    if purify_pbar is False:
        # Update and set description
        pbar.update(1)
        pbar.set_description(f'Poisoned Data Saved to {save_dir}')

Files already downloaded and verified


Purifying Data:   0%|          | 0/391 [00:00<?, ?it/s]